<a href="https://colab.research.google.com/github/august-ehrlich/delex-lm/blob/main/CompLing2ProjectFinal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformer_lens;
!pip install torch
!pip install datasets

In [ ]:
import datasets
import torch
import transformer_lens
import transformer_lens.utils as utils
from transformer_lens import HookedTransformer
from collections import Counter
from typing import List

In [ ]:
class UDSurfaceTokenizer:
    def __init__(self):
        # 1. Define mandatory special tokens for causal modeling
        self.PAD_TOKEN = "<pad>"  # Used to make sequences the same length in a batch
        self.UNK_TOKEN = "<unk>"  # Used if the model sees a bundle it never saw in training
        self.EOS_TOKEN = "<eos>"  # End of sequence (already in your dataset)

        self.special_tokens = [self.PAD_TOKEN, self.UNK_TOKEN, self.EOS_TOKEN]

        # 2. Initialize the vocabulary mappings
        self.vocab_to_id = {token: idx for idx, token in enumerate(self.special_tokens)}
        self.id_to_vocab = {idx: token for idx, token in enumerate(self.special_tokens)}

        self.vocab_size = len(self.vocab_to_id)

    def train_on_corpus(self, corpus: List[str], min_freq: int = 1):
        """
        Reads the training data, finds all unique tokens, and assigns them an ID.
        `corpus` should be a list of your dataset's strings.
        """
        token_counts = Counter()

        # Count frequencies of every space-separated block
        for sentence in corpus:
            tokens = sentence.strip().split()
            token_counts.update(tokens)

        # Add to vocabulary if it meets the minimum frequency threshold
        # (Filtering out rare tokens can help prevent the model from memorizing noise)
        for token, count in token_counts.items():
            if token not in self.vocab_to_id and count >= min_freq:
                idx = len(self.vocab_to_id)
                self.vocab_to_id[token] = idx
                self.id_to_vocab[idx] = token

        self.vocab_size = len(self.vocab_to_id)
        print(f"Vocabulary built! Total unique tokens: {self.vocab_size}")

    def encode(self, text: str, add_eos: bool = False) -> List[int]:
        """
        Converts a string of UD bundles/literals into a list of integer IDs.
        """
        tokens = text.strip().split()

        # Map token to ID, fallback to <unk> if it doesn't exist
        token_ids = [self.vocab_to_id.get(token, self.vocab_to_id[self.UNK_TOKEN]) for token in tokens]

        if add_eos:
            token_ids.append(self.vocab_to_id[self.EOS_TOKEN])

        return token_ids

    def decode(self, token_ids: List[int]) -> str:
        """
        Converts a list of integer IDs back into the readable UD string format.
        """
        tokens = [self.id_to_vocab.get(idx, self.UNK_TOKEN) for idx in token_ids]
        return " ".join(tokens)

    def save(self, filepath: str):
        """Saves the vocabulary dictionary to disk."""
        import json
        with open(filepath, 'w', encoding='utf-8') as f:
            json.dump(self.vocab_to_id, f, ensure_ascii=False, indent=2)

    def load(self, filepath: str):
        """Loads a pre-computed vocabulary dictionary."""
        import json
        with open(filepath, 'r', encoding='utf-8') as f:
            self.vocab_to_id = json.load(f)
        self.id_to_vocab = {int(idx): token for token, idx in self.vocab_to_id.items()}
        self.vocab_size = len(self.vocab_to_id)